In [1]:
!uv pip install --upgrade pip
!uv pip install huggingface_hub accelerate scikit-learn
!uv pip uninstall torchvision pandas
!uv pip uninstall transformers tokenizers accelerate
!uv pip install "transformers==4.56.0" "protobuf==5.29.3"
!uv pip install torch datasets
!uv pip install tqdm wandb
!uv pip install bitsandbytes accelerate hf_transfer
!uv pip install --force-reinstall --no-cache-dir "numpy==2.0"

# Install PyTorch/XLA for TPU (Kaggle)
# !uv pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

Using Python 3.12.12 environment at: /usr
Resolved 1 package in 216ms                                          
Prepared 1 package in 131ms                                              
Uninstalled 1 package in 206ms
Installed 1 package in 24ms                                 
 - pip==24.1.2
 + pip==26.1
Using Python 3.12.12 environment at: /usr
Audited 3 packages in 258ms
Using Python 3.12.12 environment at: /usr
Uninstalled 2 packages in 728ms
 - pandas==2.3.3
 - torchvision==0.25.0+cu128
Using Python 3.12.12 environment at: /usr
Uninstalled 3 packages in 1.08s
 - accelerate==1.12.0
 - tokenizers==0.22.2
 - transformers==5.0.0
Using Python 3.12.12 environment at: /usr
Resolved 19 packages in 311ms                                        
Prepared 4 packages in 672ms                                             
Uninstalled 2 packages in 44ms
Installed 4 packages in 100ms                               
 - huggingface-hub==1.4.1
 + huggingface-hub==0.36.2
 - protobuf==5.29.5
 + protobuf=

In [2]:
import os
import re
import logging
import sys
import subprocess
from pathlib import Path
import json

import torch
import torch.nn as nn
# import torch_xla
# import torch_xla.core.xla_model as xm

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

from IPython import get_ipython
import wandb
from typing import Dict, Tuple

def configure_environment_paths():
    try:
        if "google.colab" in str(get_ipython()):
            print("✅ Environment: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            print("✅ Environment: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            print("⚠️ Environment: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        print("⚠️ Non-interactive session. Using local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"
    os.makedirs(base_output_path, exist_ok=True)
    print(f"📂 Data Path: {base_data_path}")
    print(f"📦 Output Path: {base_output_path}")
    return base_data_path, base_output_path, environment_name

def load_secret(key_name: str, env_name: str) -> str | None:
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env_name}' environment...")
    try:
        if env_name == "colab":
            from google.colab import userdata
            secret_value = userdata.get(key_name)
        elif env_name == "kaggle":
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"⚠️ Secret '{key_name}' not found in the {env_name} environment.")
            return None
        print(f"✅ Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"❌ An error occurred while loading secret '{key_name}': {e}")
        return None

def print_system_info():
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch
        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
        # TPU detection
        try:
            import torch_xla
            print(f"PyTorch/XLA version: {torch_xla.__version__}")
            print(f"TPU cores: {xm.xrt_world_size()}")
        except:
            pass
    except ImportError:
        print("PyTorch not installed")

INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle
print_system_info()

os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY", ENV_NAME)
os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN", ENV_NAME)
GITHUB_TOKEN = load_secret("GITHUB_TOKEN", ENV_NAME)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import wandb
import huggingface_hub
wandb.login()
huggingface_hub.login(token=os.environ["HF_TOKEN"])


2026-04-28 08:36:30.978891: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777365391.375370      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777365391.484769      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777365392.450030      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777365392.450071      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777365392.450073      55 computation_placer.cc:177] computation placer alr

✅ Environment: Kaggle
📂 Data Path: /kaggle/input/
📦 Output Path: /kaggle/working/

🔧 System Information
Python version: 3.12.12
PyTorch version: 2.10.0+cu128
CUDA version: 12.8
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Attempting to load secret 'WANDB_API_KEY' from 'kaggle' environment...
✅ Successfully loaded secret 'WANDB_API_KEY'.
Attempting to load secret 'HF_TOKEN' from 'kaggle' environment...
✅ Successfully loaded secret 'HF_TOKEN'.
Attempting to load secret 'GITHUB_TOKEN' from 'kaggle' environment...
✅ Successfully loaded secret 'GITHUB_TOKEN'.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: dungngocpham171 (dungngocpham171-university-of-science) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
def download_from_hf(
    repo_id: str,
    local_dir: str = "ckpt",
    allow_patterns=None,
    force_download: bool = False,
    repo_type: str = "model",
):
    if allow_patterns is None:
        allow_patterns = ["*.safetensors"]
    print(f"📥 Downloading checkpoint from Hugging Face Hub to '{local_dir}'...\n")
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        repo_type=repo_type,
        allow_patterns=allow_patterns,
        force_download=force_download,
    )
    print("\n✅ Download complete!")
    print(f"\n📂 Files in {local_dir}/:")
    for file in os.listdir(local_dir):
        if file.endswith(".safetensors"):
            size = os.path.getsize(os.path.join(local_dir, file)) / (1024**2)
            print(f"  ✓ {file} ({size:.2f} MB)")

In [4]:
!rm -rf ckpt
download_from_hf(
    repo_id="dzungpham/graphcodebert-code-classification",
    local_dir="checkpoints",
    allow_patterns=["graphcodebert-base-lowLR-highBatchSize/checkpoint-1022*"],
)

📥 Downloading checkpoint from Hugging Face Hub to 'checkpoints'...



Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

graphcodebert-base-lowLR-highBatchSize/c(…):   0%|          | 0.00/499M [00:00<?, ?B/s]

graphcodebert-base-lowLR-highBatchSize/c(…):   0%|          | 0.00/4.74M [00:00<?, ?B/s]

graphcodebert-base-lowLR-highBatchSize/c(…):   0%|          | 0.00/1.38k [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

graphcodebert-base-lowLR-highBatchSize/c(…):   0%|          | 0.00/5.91k [00:00<?, ?B/s]

trainer_state.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]


✅ Download complete!

📂 Files in checkpoints/:


In [5]:
# ============================================================================
# 1. DATA LAYER: Load and preprocess datasets
# ============================================================================

import random
import os
import re
import logging
import numpy as np
from typing import Dict, Tuple
from datasets import load_dataset, Dataset, concatenate_datasets

def setup_logger(output_dir: str, name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(console_handler)
    os.makedirs(output_dir, exist_ok=True)
    log_path = Path(output_dir) / f"{name}.log"
    file_handler = logging.FileHandler(log_path, mode="w")
    file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(file_handler)
    return logger
logger = setup_logger(".", "notebook")

def set_seed(seed: int):
    """Set all random seeds for reproducibility (including TPU if available)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # For TPU, seed is handled by torch.manual_seed
    os.environ['PYTHONHASHSEED'] = str(seed)

def preprocess_code(code_str: str, seed: int = None) -> str:
    """Normalize code string with semantic-preserving perturbations for training.
       Uses the global random state; seed must be set before calling."""
    code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
    code_str = re.sub(r"\r\n|\r", "\n", code_str)

    # 20% chance to replace 4 spaces with tabs
    if random.random() < 0.2:
        code_str = code_str.replace("    ", "\t")
    # 20% chance to replace tabs with 2 spaces
    elif random.random() < 0.2:
        code_str = code_str.replace("    ", "  ")

    code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
    code_str = re.sub(r"\n{3,}", "\n\n", code_str)
    code_str = re.sub(r"[ \t]+", " ", code_str)
    return code_str.strip()

def tokenize_function(examples: Dict, tokenizer, max_length: int = 512) -> Dict:
    """Tokenize code examples."""
    codes = [preprocess_code(c) for c in examples["code"]]
    return tokenizer(codes, truncation=True, max_length=max_length, padding=False)

def upsample_dataset(dataset: Dataset) -> Dataset:
    """Upsample the minority class to match the majority class size."""
    labels = np.array(dataset["label"])
    class_counts = np.bincount(labels)
    majority_class = np.argmax(class_counts)
    minority_class = np.argmin(class_counts)

    logger.info(f"Initial distribution: Class 0: {class_counts[0]}, Class 1: {class_counts[1]}")

    diff = class_counts[majority_class] - class_counts[minority_class]

    if diff > 0:
        minority_indices = np.where(labels == minority_class)[0]
        upsample_indices = np.random.choice(minority_indices, size=diff, replace=True)
        upsampled_data = dataset.select(upsample_indices)
        dataset = concatenate_datasets([dataset, upsampled_data])

        new_counts = np.bincount(np.array(dataset["label"]))
        logger.info(f"Balanced distribution: Class 0: {new_counts[0]}, Class 1: {new_counts[1]}")

    return dataset.shuffle(seed=42)

def load_datasets(tokenizer, max_length: int = 512, aug=None) -> Tuple[Dataset, Dataset]:
    """Load and tokenize train and validation datasets with upsampling."""
    logger.info("Loading datasets from Hugging Face Hub...")
    train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
    val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="test")

    logger.info(f"Train samples: {len(train_dataset)}")
    # Uncomment if upsampling desired:
    train_dataset = upsample_dataset(train_dataset)

    logger.info(f"Final samples: Train: {len(train_dataset)}, Val: {len(val_dataset)}")

    if aug is not None:
        logger.info("Applying data augmentation before tokenization...")
        def augment_batch(examples):
            examples["code"] = [aug(c) for c in examples["code"]]
            return examples
        train_dataset = train_dataset.map(
            augment_batch,
            batched=True,
            batch_size=512,
            desc="Augmenting train",
            num_proc=os.cpu_count(),
        )

    # Tokenize
    logger.info("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in train_dataset.column_names],
        desc="Tokenizing train",
        num_proc=os.cpu_count(),
    )

    val_dataset = val_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in val_dataset.column_names],
        desc="Tokenizing val",
        num_proc=os.cpu_count(),
    )

    # Rename label column
    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")

    return train_dataset, val_dataset

def compute_metrics_fn(eval_pred):
    """Compute precision, recall, F1, and accuracy for evaluation."""
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }


In [6]:
# ============================================================================
# 2. MODEL LAYER: Load pretrained CodeBERT and tokenizer
# ============================================================================

def load_model_and_tokenizer(cfg):
    """Load pretrained model and tokenizer from Hugging Face, applying dropout rates from cfg."""
    from transformers import AutoConfig

    logger.info(f"Loading tokenizer from: {cfg.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    logger.info(f"Loading config from: {cfg.model_name}")
    config = AutoConfig.from_pretrained(cfg.model_name)
    config.num_labels = cfg.num_labels
    config.problem_type = "single_label_classification"

    # Override dropout rates from cfg
    config.hidden_dropout_prob = cfg.hidden_dropout_prob
    config.attention_probs_dropout_prob = cfg.attention_probs_dropout_prob
    if hasattr(config, 'classifier_dropout'):
        config.classifier_dropout = cfg.classifier_dropout

    logger.info(f"Loading model with custom config: hidden_dropout={config.hidden_dropout_prob}, "
                f"attention_dropout={config.attention_probs_dropout_prob}, "
                f"classifier_dropout={getattr(config, 'classifier_dropout', 'N/A')}")

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        config=config,
    )
    logger.info(f"Model loaded successfully")
    logger.info(f"Model config - num_labels: {model.config.num_labels}, hidden_size: {model.config.hidden_size}")
    return model, tokenizer

def freeze_base_model(model):
    """
    Freeze all base model parameters, keeping only classifier head trainable.
    """
    for name, param in model.named_parameters():
        if "classifier" not in name and "cls" not in name.lower():
            param.requires_grad = False
    for name, param in model.named_parameters():
        if "classifier" in name or "cls" in name.lower():
            param.requires_grad = True
    logger.info("Base model frozen - only classifier head is trainable")

In [7]:
def log_training_config(cfg, logger):
    logger.info("===== Training Configuration =====")
    for field_name, field_value in cfg.__dataclass_fields__.items():
        value = getattr(cfg, field_name)
        if field_name == "device":
            continue
        logger.info(f"{field_name:20} : {value}")
    logger.info("=================================")

def log_model_architecture(model, tokenizer, logger):
    logger.info("===== Model Architecture =====")
    logger.info("\n" + model.__repr__())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    logger.info("===== Parameter Summary =====")
    logger.info(f"Total Parameters:         {total_params:,}")
    logger.info(f"Trainable Parameters:     {trainable_params:,}")
    logger.info(f"Non-trainable Parameters: {non_trainable_params:,}")
    logger.info("===== Tokenizer Summary =====")
    logger.info(f"Vocab size: {len(tokenizer)} | Special tokens: {tokenizer.all_special_tokens}")
    logger.info("===== End of Architecture Log =====")

In [8]:
# ============================================================================
# CONFIG LOGGING CALLBACK: Save hyperparameters at each checkpoint
# ============================================================================
from transformers import TrainerCallback
class ConfigLoggingCallback(TrainerCallback):
    """
    Callback to save hyperparameters and configuration to each checkpoint.

    At each save_steps interval, writes a comprehensive config JSON file to
    the checkpoint directory containing all TrainConfig and TrainingArguments.
    """

    def __init__(self, train_config):
        """
        Args:
            train_config: TrainConfig instance with all hyperparameters
        """
        self.train_config = train_config

    def on_save(self, args, state, control, **kwargs):
        """Called after a checkpoint is saved."""
        checkpoint_dir = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")

        if not os.path.exists(checkpoint_dir):
            logger.warning(f"Expected checkpoint directory not found: {checkpoint_dir}")
            return

        import json
        config_dict = self._prepare_config_dict(args, state)

        config_path = os.path.join(checkpoint_dir, "config_hyperparams.json")
        try:
            with open(config_path, "w") as f:
                json.dump(config_dict, f, indent=2, default=str)
            logger.info(f"✅ Saved hyperparameters to {config_path}")
        except Exception as e:
            logger.error(f"❌ Failed to save config to {config_path}: {e}")

    def _prepare_config_dict(self, training_args, state):
        """Prepare comprehensive config dictionary with all parameters."""
        import json

        # Extract TrainConfig as dictionary
        train_cfg_dict = {
            "model_name": self.train_config.model_name,
            "num_epochs": self.train_config.num_epochs,
            "batch_size": self.train_config.batch_size,
            "learning_rate": self.train_config.learning_rate,
            "max_length": self.train_config.max_length,
            "num_labels": self.train_config.num_labels,
            "loss_type": self.train_config.loss_type,
            "focal_alpha": self.train_config.focal_alpha,
            "focal_gamma": self.train_config.focal_gamma,
            "r_drop_alpha": self.train_config.r_drop_alpha,
            "infonce_temperature": self.train_config.infonce_temperature,
            "infonce_weight": self.train_config.infonce_weight,
            "label_smoothing": self.train_config.label_smoothing,
            "adversarial_epsilon": self.train_config.adversarial_epsilon,
            "use_swa": self.train_config.use_swa,
            "swa_start_epoch": self.train_config.swa_start_epoch,
            "swa_lr": self.train_config.swa_lr,
            "data_augmentation": self.train_config.data_augmentation,
            "aug_rename_prob": self.train_config.aug_rename_prob,
            "aug_format_prob": self.train_config.aug_format_prob,
            "freeze_base": self.train_config.freeze_base,
            "seed": self.train_config.seed,
            "use_wandb": self.train_config.use_wandb,
            "mixup_alpha": self.train_config.mixup_alpha,
            "low_pass_keep_ratio": self.train_config.low_pass_keep_ratio,
            "freq_consistency_weight": self.train_config.freq_consistency_weight,
        }

        # Extract key TrainingArguments
        training_args_dict = {
            "output_dir": training_args.output_dir,
            "num_train_epochs": training_args.num_train_epochs,
            "per_device_train_batch_size": training_args.per_device_train_batch_size,
            "per_device_eval_batch_size": training_args.per_device_eval_batch_size,
            "learning_rate": training_args.learning_rate,
            "warmup_steps": training_args.warmup_steps,
            "weight_decay": training_args.weight_decay,
            "logging_steps": training_args.logging_steps,
            "eval_steps": training_args.eval_steps,
            "save_steps": training_args.save_steps,
            "metric_for_best_model": training_args.metric_for_best_model,
            "greater_is_better": training_args.greater_is_better,
            "save_total_limit": training_args.save_total_limit,
            "fp16": training_args.fp16,
            "seed": training_args.seed,
        }

        # Extract training state
        training_state_dict = {
            "global_step": state.global_step,
            "epoch": state.epoch,
            "best_metric": state.best_metric,
            "best_model_checkpoint": state.best_model_checkpoint,
        }

        # Combine all sections
        return {
            "train_config": train_cfg_dict,
            "training_arguments": training_args_dict,
            "training_state": training_state_dict,
        }


In [17]:
# =============================================================================
# 3. TRAINING ENGINE: HF Trainer with MixCode + FFT low-pass + frequency consistency
# =============================================================================

import os
import logging
import random
import numpy as np
from dataclasses import dataclass, field, asdict
from typing import Callable, Optional, Tuple, List, Dict
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    PreTrainedTokenizer,
)

# ------------------------------
# FFT low-pass filter function
# ------------------------------
def low_pass_filter_embeddings(embeddings, keep_ratio=0.5):
    """
    Apply low‑pass filter to embeddings in the frequency domain.
    Args:
        embeddings: torch.Tensor of shape (batch, seq_len, hidden_dim)
        keep_ratio: fraction of low frequencies to keep (0.0 to 1.0)
    Returns:
        filtered_embeddings: same shape as input
    """
    batch, seq_len, hidden = embeddings.shape
    # FFT along sequence dimension (dim=1)
    fft = torch.fft.rfft(embeddings, dim=1, norm='ortho')
    keep_len = max(1, int(seq_len * keep_ratio))
    fft[:, keep_len:] = 0.0
    filtered = torch.fft.irfft(fft, n=seq_len, dim=1, norm='ortho')
    return filtered

# ------------------------------
# Enhanced CodeAugmentation with mixup parameters
# ------------------------------
class CodeAugmentation:
    def __init__(self, rename_prob=0.3, format_prob=0.3, mixup_alpha=1.0):
        self.rename_prob = rename_prob
        self.format_prob = format_prob
        self.mixup_alpha = mixup_alpha   # Beta distribution parameter for MixCode

    def __call__(self, text: str) -> str:
        text = str(text)
        if random.random() < self.rename_prob:
            import re
            tokens = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', text)
            keywords = {'def', 'return', 'if', 'else', 'for', 'while', 'import',
                        'from', 'as', 'with', 'try', 'except', 'class', 'lambda',
                        'and', 'or', 'not', 'True', 'False', 'None'}
            identifiers = [t for t in set(tokens) if t not in keywords and len(t) > 1]
            if identifiers:
                rename_map = {id_: f"var_{random.randint(1000,9999)}" for id_ in identifiers[:5]}
                for old, new in rename_map.items():
                    text = text.replace(old, new)
        if random.random() < self.format_prob:
            lines = text.split('\n')
            new_lines = []
            for line in lines:
                new_lines.append(line)
                if random.random() < 0.1:
                    new_lines.append('')
            text = '\n'.join(new_lines)
            if random.random() < 0.2:
                text = '\n'.join(' ' + line if random.random() < 0.1 else line
                                 for line in text.split('\n'))
        return text

# ------------------------------
# Loss function factories (unchanged)
# ------------------------------
def get_label_smoothed_cross_entropy(smoothing: float) -> Callable:
    def loss_fn(outputs, labels, **_):
        logits = outputs.logits
        n_classes = logits.size(-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits).fill_(smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
        return -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()
    return loss_fn

def get_focal_loss(alpha: float, gamma: float, smoothing: float = 0.0) -> Callable:
    def focal_loss(outputs, labels, **_):
        logits = outputs.logits
        if smoothing > 0:
            n_classes = logits.size(-1)
            smooth_targets = torch.zeros_like(logits).fill_(smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
            ce = -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1)
        else:
            ce = F.cross_entropy(logits, labels, reduction="none")
        pt = torch.exp(-ce)
        loss = alpha * (1 - pt) ** gamma * ce
        return loss.mean()
    return focal_loss

def get_infonce_loss(temperature: float, weight: float) -> Callable:
    def infonce_loss(outputs, labels, **_):
        reps = outputs.hidden_states[-1]
        reps = F.normalize(reps, dim=-1)
        sim_matrix = torch.mm(reps, reps.t()) / temperature
        target = torch.arange(reps.size(0), device=reps.device)
        loss = F.cross_entropy(sim_matrix, target)
        return weight * loss
    return infonce_loss

# ------------------------------
# RobustTrainer with MixCode and frequency consistency
# ------------------------------
class RobustTrainer(Trainer):
    def __init__(
        self,
        *args,
        loss_type: str = "ce",
        r_drop_alpha: float = 4.0,
        compute_loss_fn: Optional[Callable] = None,
        label_smoothing: float = 0.0,
        adversarial_epsilon: float = 0.0,
        use_swa: bool = False,
        swa_start_epoch: int = 2,
        swa_lr: float = 1e-5,
        mixup_alpha: float = 1.0,
        low_pass_keep_ratio: float = 0.5,
        freq_consistency_weight: float = 0.1,
        hyperparams: Optional[dict] = None,          # <-- NEW
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.loss_type = loss_type
        self.r_drop_alpha = r_drop_alpha
        self.compute_loss_fn = compute_loss_fn
        self.label_smoothing = label_smoothing
        self.adversarial_epsilon = adversarial_epsilon
        self.use_swa = use_swa
        self.swa_start_epoch = swa_start_epoch
        self.swa_lr = swa_lr
        self.swa_model = None
        self.swa_scheduler = None
        self._swa_started = False
        # New parameters
        self.mixup_alpha = mixup_alpha
        self.low_pass_keep_ratio = low_pass_keep_ratio
        self.freq_consistency_weight = freq_consistency_weight
        self.hyperparams = hyperparams or {}

    def compute_loss(self, model, inputs, num_items_in_batch=64, return_outputs=False):
        labels = inputs.pop("labels", None)

        # ---- R-Drop (unchanged) ----
        if self.loss_type == "r-drop" and model.training and labels is not None:
            out1 = model(**inputs)
            out2 = model(**inputs)
            ce1 = F.cross_entropy(out1.logits, labels)
            ce2 = F.cross_entropy(out2.logits, labels)
            ce = (ce1 + ce2) / 2
            p = F.log_softmax(out1.logits, dim=-1)
            q = F.log_softmax(out2.logits, dim=-1)
            kl = (F.kl_div(p, F.softmax(out2.logits, dim=-1), reduction="batchmean") +
                  F.kl_div(q, F.softmax(out1.logits, dim=-1), reduction="batchmean")) / 2
            loss = ce + self.r_drop_alpha * kl
            return (loss, out1) if return_outputs else loss

        # ---- MixCode (if enabled) ----
        if self.mixup_alpha > 0 and model.training and labels is not None:
            batch_size = inputs['input_ids'].size(0)
            device = inputs['input_ids'].device
            shuffle_idx = torch.randperm(batch_size, device=device)
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)

            # Get embedding layer (GraphCodeBERT uses roberta)
            emb_layer = model.roberta.embeddings.word_embeddings
            orig_embeds = emb_layer(inputs['input_ids'])
            shuffled_embeds = emb_layer(inputs['input_ids'][shuffle_idx])

            mixed_embeds = lam * orig_embeds + (1 - lam) * shuffled_embeds
            mixed_attn = inputs['attention_mask']

            outputs = model(inputs_embeds=mixed_embeds, attention_mask=mixed_attn)

            # Soft labels
            n_classes = model.config.num_labels
            one_hot = F.one_hot(labels, n_classes).float()
            shuffled_labels = labels[shuffle_idx]
            one_hot_shuffled = F.one_hot(shuffled_labels, n_classes).float()
            mixed_labels = lam * one_hot + (1 - lam) * one_hot_shuffled

            loss = -(mixed_labels * F.log_softmax(outputs.logits, dim=-1)).sum(dim=-1).mean()
        else:
            outputs = model(**inputs)
            if labels is None:
                loss = outputs.loss if hasattr(outputs, "loss") else outputs[0]
            else:
                if self.compute_loss_fn:
                    loss = self.compute_loss_fn(outputs, labels)
                else:
                    if self.label_smoothing > 0 and self.loss_type == "ce":
                        loss_fn = get_label_smoothed_cross_entropy(self.label_smoothing)
                        loss = loss_fn(outputs, labels)
                    else:
                        loss = F.cross_entropy(outputs.logits, labels)

        # ---- Frequency-domain consistency loss (FFT low-pass) ----
        if self.freq_consistency_weight > 0 and model.training and labels is not None:
            emb_layer = model.roberta.embeddings.word_embeddings
            orig_embeds = emb_layer(inputs['input_ids'])
            filtered_embeds = low_pass_filter_embeddings(orig_embeds, self.low_pass_keep_ratio)
            filtered_outputs = model(inputs_embeds=filtered_embeds, attention_mask=inputs['attention_mask'])
            kl = F.kl_div(
                F.log_softmax(outputs.logits, dim=-1),
                F.softmax(filtered_outputs.logits, dim=-1),
                reduction='batchmean'
            )
            loss = loss + self.freq_consistency_weight * kl

        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch):
        """Override to add FGM adversarial training and SWA updates."""
        # Standard forward + backward (first pass)
        loss = super().training_step(model, inputs)

        # ---- FGM Adversarial Training ----
        if self.adversarial_epsilon > 0 and model.training:
            # Clone inputs to avoid modifying original
            adv_inputs = {k: v.clone() if torch.is_tensor(v) else v for k, v in inputs.items()}
            labels = adv_inputs.get("labels", None)
            if labels is not None:
                # 1. Clean backward already done, so gradients exist.
                # 2. Get embeddings (handle different model architectures)
                if hasattr(model, 'roberta'):
                    embeddings = model.roberta.embeddings.word_embeddings
                elif hasattr(model, 'bert'):
                    embeddings = model.bert.embeddings.word_embeddings
                else:
                    embeddings = None
                if embeddings is not None:
                    grad = embeddings.weight.grad
                    if grad is not None:
                        perturbation = self.adversarial_epsilon * grad.sign()
                        original_emb = embeddings.weight.data.clone()
                        embeddings.weight.data.add_(perturbation)

                        # Forward with perturbed embeddings
                        adv_outputs = model(**{k:v for k,v in adv_inputs.items() if k != 'labels'})
                        if self.compute_loss_fn:
                            adv_loss = self.compute_loss_fn(adv_outputs, labels)
                        else:
                            if self.label_smoothing > 0 and self.loss_type == "ce":
                                loss_fn = get_label_smoothed_cross_entropy(self.label_smoothing)
                                adv_loss = loss_fn(adv_outputs, labels)
                            else:
                                adv_loss = F.cross_entropy(adv_outputs.logits, labels)

                        # Backward for adversarial loss (accumulate)
                        self.accelerator.backward(adv_loss)

                        # Restore embeddings
                        embeddings.weight.data = original_emb

        # ---- SWA Initialization and Update ----
        if self.use_swa and not self._swa_started and self.state.epoch >= self.swa_start_epoch:
            self._swa_started = True
            self.swa_model = AveragedModel(model)
            self.swa_scheduler = SWALR(self.optimizer, swa_lr=self.swa_lr)
            self.log({"SWA": "started"})

        if self._swa_started and self.swa_model is not None:
            self.swa_model.update_parameters(model)

        return loss

    def optimizer_step(self, model, optimizer, optimizer_idx=None):
        super().optimizer_step(model, optimizer, optimizer_idx)
        if self._swa_started and self.swa_scheduler is not None:
            self.swa_scheduler.step()

    def _unwrap_model(self, model):
        # Handles DataParallel, DistributedDataParallel, etc.
        return model.module if hasattr(model, "module") else model

    def save_model(self, output_dir: Optional[str] = None, _internal_call: bool = False):
        if self._swa_started and self.swa_model is not None:
            if self.train_dataset is not None:
                train_loader = self.get_train_dataloader()
                update_bn(train_loader, self.swa_model)
            model_to_save = self.swa_model.module if hasattr(self.swa_model, 'module') else self.swa_model
            model_to_save.save_pretrained(output_dir or self.args.output_dir)
        else:
            super().save_model(output_dir, _internal_call)
        # ---- NEW: Save hyperparams.json ----
        hyperparams_path = os.path.join(output_dir, "hyperparams.json")

        def serialize(obj):
            """Convert non‑JSON‑serializable objects to strings."""
            if isinstance(obj, Path):
                return str(obj)
            if isinstance(obj, torch.device):
                return str(obj)
            if hasattr(obj, "value"):  # Enum
                return obj.value
            try:
                json.dumps(obj)
                return obj
            except (TypeError, OverflowError):
                return str(obj)

        safe_dict = {k: serialize(v) for k, v in self.hyperparams.items()}
        with open(hyperparams_path, "w", encoding="utf-8") as f:
            json.dump(safe_dict, f, indent=4, ensure_ascii=False)

        print(f"Hyperparameters saved to {hyperparams_path}")
# ------------------------------
# Updated TrainConfig with new parameters
# ------------------------------
@dataclass
class TrainConfig:
    model_name: str = "microsoft/graphcodebert-base"
    output_dir: str = "./graphcodebert"
    num_epochs: int = 3
    max_steps: int = -1
    batch_size: int = 16
    learning_rate: float = 2e-5
    max_length: int = 512
    num_labels: int = 2
    use_wandb: bool = False
    freeze_base: bool = False
    loss_type: str = "r-drop"
    focal_alpha: float = 1.0
    focal_gamma: float = 2.0
    r_drop_alpha: float = 4.0
    infonce_temperature: float = 0.07
    infonce_weight: float = 0.5
    seed: int = 42
    wandb_run_name: str = "beyond-the-infinity"
    resume_from_checkpoint: str = None

    save_steps: int = 100
    eval_steps: int = 100
    logging_steps: int = 2

    label_smoothing: float = 0.1
    adversarial_epsilon: float = 0.5
    use_swa: bool = True
    swa_start_epoch: int = 2
    swa_lr: float = 1e-5
    data_augmentation: bool = True
    aug_rename_prob: float = 0.3
    aug_format_prob: float = 0.3

    # New parameters for MixCode and frequency domain
    mixup_alpha: float = 1.0
    low_pass_keep_ratio: float = 0.5
    freq_consistency_weight: float = 0.1

    hidden_dropout_prob: float = 0.3
    attention_probs_dropout_prob: float = 0.3
    classifier_dropout: float = 0.3

    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    def copy(self) -> "TrainConfig":
        """Return a deep copy of the config."""
        return TrainConfig(**{k: v for k, v in asdict(self).items()})

# ------------------------------
# Main training pipeline (updated)
# ------------------------------
def train_pipeline(cfg: TrainConfig):
    os.makedirs(cfg.output_dir, exist_ok=True)
    logger = setup_logger(cfg.output_dir, "training")

    set_seed(cfg.seed)

    log_training_config(cfg, logger)

    # Load model & tokenizer
    model, tokenizer = load_model_and_tokenizer(cfg)
    # Handle TPU device if available
    if "xla" in str(cfg.device) or (hasattr(torch, 'xla') and xm.xrt_world_size() > 1):
        device = xm.xla_device()
        logger.info("Using TPU device")
    else:
        device = cfg.device
    model.to(device)
    logger.info(f"Model placed on {device}")

    if cfg.freeze_base:
        freeze_base_model(model)

    log_model_architecture(model, tokenizer, logger)

    if cfg.loss_type == "infonce":
        model.config.output_hidden_states = True
        logger.info("Enabled hidden states for InfoNCE loss.")

    # Data augmentation (now includes mixup_alpha, but that's handled in trainer)
    aug = None
    if cfg.data_augmentation:
        aug = CodeAugmentation(rename_prob=cfg.aug_rename_prob,
                               format_prob=cfg.aug_format_prob,
                               mixup_alpha=cfg.mixup_alpha)
        logger.info(f"Data augmentation enabled (rename={cfg.aug_rename_prob}, format={cfg.aug_format_prob})")

    train_dataset, val_dataset = load_datasets(tokenizer, cfg.max_length, aug=aug)

    if cfg.use_wandb:
        try:
            import wandb
            os.environ["WANDB_MODE"] = "online"
        except Exception as e:
            logger.warning(f"W&B import failed ({e}); proceeding without it.")
            cfg.use_wandb = False

    steps_per_epoch = max(1, len(train_dataset) // cfg.batch_size)
    total_steps = cfg.num_epochs * steps_per_epoch
    warmup_steps = max(1, int(total_steps * 0.1))

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_epochs,
        max_steps=cfg.max_steps,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size * 2,
        learning_rate=cfg.learning_rate,
        warmup_steps=warmup_steps,
        weight_decay=0.1,
        logging_strategy="steps",
        logging_steps=cfg.logging_steps,
        eval_strategy="steps",
        eval_steps=cfg.eval_steps,
        save_strategy="steps",
        save_steps=cfg.save_steps,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=5,
        report_to=["wandb"] if cfg.use_wandb else [],
        run_name=cfg.wandb_run_name if cfg.use_wandb else None,
        dataloader_pin_memory=torch.cuda.is_available(),
        seed=cfg.seed,
        fp16=False if "xla" in str(device) else True,  # TPU uses bf16, not fp16
        bf16=True if "xla" in str(device) else False,

        lr_scheduler_type="cosine_with_restarts",                     # Use cosine decay with warmup
        lr_scheduler_kwargs={"num_cycles": 2},        # Optional: adjust number of cycles (default 0.5)

    )

    data_collator = DataCollatorWithPadding(tokenizer)

    compute_loss_fn = None
    if cfg.loss_type == "focal":
        compute_loss_fn = get_focal_loss(cfg.focal_alpha, cfg.focal_gamma, smoothing=cfg.label_smoothing if cfg.loss_type=="ce" else 0.0)
    elif cfg.loss_type == "infonce":
        compute_loss_fn = get_infonce_loss(cfg.infonce_temperature, cfg.infonce_weight)

    trainer = RobustTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            ConfigLoggingCallback(train_config=cfg),
        ],
        loss_type=cfg.loss_type,
        r_drop_alpha=cfg.r_drop_alpha,
        compute_loss_fn=compute_loss_fn,
        label_smoothing=cfg.label_smoothing,
        adversarial_epsilon=cfg.adversarial_epsilon,
        use_swa=cfg.use_swa,
        swa_start_epoch=cfg.swa_start_epoch,
        swa_lr=cfg.swa_lr,
        mixup_alpha=cfg.mixup_alpha,
        low_pass_keep_ratio=cfg.low_pass_keep_ratio,
        freq_consistency_weight=cfg.freq_consistency_weight,
        hyperparams=asdict(cfg),
    )

    logger.info("=== Starting training with MixCode + FFT low-pass consistency ===")
    try:
        trainer.train(resume_from_checkpoint=cfg.resume_from_checkpoint)
        logger.info("Training completed successfully.")
    except KeyboardInterrupt:
        logger.info("Training interrupted by user.")
        raise  # or exit
    except Exception as e:
        logger.error(f"Training failed: {e}")
        raise
    finally:
        wandb.finish()

    final_model = trainer.swa_model if (trainer._swa_started and trainer.swa_model) else model
    final_model.save_pretrained(os.path.join(cfg.output_dir, "final_model"))
    tokenizer.save_pretrained(os.path.join(cfg.output_dir, "final_model"))
    logger.info(f"Final model saved to {os.path.join(cfg.output_dir, 'final_model')}")

    return trainer, final_model, tokenizer

In [10]:
# if __name__ == "__main__":
#     import gc
#     import itertools
#     import torch

#     # -------------------------------------------------------------------------
#     # 1. Define the hyperparameter grid (major params with 2–3 values each)
#     # -------------------------------------------------------------------------
#     param_grid = {
#         "batch_size": [16, 32, 64],
#         "learning_rate": [5e-6, 1e-5, 5e-5],
#         "r_drop_alpha": [4.0, 6.0, 8.0],
#         "label_smoothing": [0.2, 0.3, 0.4],
#         "mixup_alpha": [0.8, 1.0, 1.2],
#         "classifier_dropout": [0.2, 0.3, 0.4],
#     }

#     # Generate all combinations (3^6 = 729) – you may limit them
#     keys = list(param_grid.keys())
#     values = list(param_grid.values())
#     all_combinations = list(itertools.product(*values))
#     print(f"Total hyperparameter combinations: {len(all_combinations)}")

#     # Optional: limit to first N for quick testing (e.g., 10)
#     MAX_RUNS = 1000
#     if MAX_RUNS and len(all_combinations) > MAX_RUNS:
#         all_combinations = all_combinations[:MAX_RUNS]
#         print(f"Limiting to first {MAX_RUNS} runs.")

#     # -------------------------------------------------------------------------
#     # 2. Base configuration (shared across all runs)
#     # -------------------------------------------------------------------------
#     base_cfg = TrainConfig(
#         model_name="microsoft/graphcodebert-base",
#         output_dir="output_checkpoints/graphcodebert-ready-lmc/temp",  # will be overridden
#         # max_steps=50,                     # train for exactly 50 steps
#         num_epochs=1000,                  # large enough, max_steps takes precedence
#         batch_size=32,                    # will be overridden
#         learning_rate=1e-5,               # will be overridden
#         max_length=512,
#         use_wandb=False,                  # set to True if you want logging
#         freeze_base=True,
#         loss_type="r-drop",
#         r_drop_alpha=6.0,                 # will be overridden
#         label_smoothing=0.3,              # will be overridden
#         adversarial_epsilon=0.5,
#         use_swa=False,
#         swa_start_epoch=0,
#         swa_lr=1e-5,
#         data_augmentation=True,
#         aug_rename_prob=0.6,
#         aug_format_prob=0.6,
#         mixup_alpha=1.0,                  # will be overridden
#         low_pass_keep_ratio=0.5,
#         freq_consistency_weight=0.2,
#         hidden_dropout_prob=0.3,
#         attention_probs_dropout_prob=0.3,
#         classifier_dropout=0.3,           # will be overridden
#         resume_from_checkpoint=None,
#     )

#     # -------------------------------------------------------------------------
#     # 3. Run sweep
#     # -------------------------------------------------------------------------
#     sweep_root = "output_checkpoints/graphcodebert-ready-lmc"
#     os.makedirs(sweep_root, exist_ok=True)

#     for run_idx, hyperparams in enumerate(all_combinations, start=1):
#         print(f"\n=== Starting run {run_idx}/{len(all_combinations)} ===")
#         # Create a copy of the base configuration
#         cfg = base_cfg.copy()
#         # Override with specific hyperparameters
#         for k, v in zip(keys, hyperparams):
#             setattr(cfg, k, v)
#         # Set a unique output directory
#         cfg.output_dir = os.path.join(sweep_root, f"run_{run_idx}")

#         # Optionally set a random seed per run (or keep fixed)
#         set_seed(42 + run_idx)

#         # Clean GPU memory before each run
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()

#         try:
#             trainer, model, tokenizer = train_pipeline(cfg)
#             print(f"Run {run_idx} completed successfully. Model saved to {cfg.output_dir}")
#         except Exception as e:
#             print(f"Run {run_idx} failed with error: {e}")
#             continue

#     print("\nAll runs finished.")

In [ ]:
if __name__ == "__main__":
    wandb.finish()
    import os
    os.environ["WANDB_PROJECT"] = "machine-generated-code"
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    for alpha in range(1, 2, 1):
      cfg = TrainConfig(
          # ----- Model & Output -----
          model_name="microsoft/graphcodebert-base",
          output_dir="output_checkpoints/graphcodebert-best/",
          wandb_run_name=f"graphcodebert-rdrop-best",
          use_wandb=True,

          # ----- Training Basics -----
          num_epochs=2,
          # max_steps=100,
          batch_size=512,
          learning_rate=1e-6,
          save_steps=100,
          eval_steps=20,
          logging_steps=2,
          resume_from_checkpoint="checkpoints/graphcodebert-base-lowLR-highBatchSize/checkpoint-1022",

          # ----- Regularization & Dropout -----
          freeze_base=True,
          hidden_dropout_prob=0.3,
          attention_probs_dropout_prob=0.3,
          classifier_dropout=0.3,
          label_smoothing=0.5,
          swa_start_epoch=0,
          swa_lr=1e-5,

          # ----- Loss & Adversarial -----
          loss_type="r-drop",
          r_drop_alpha=6.0,
          adversarial_epsilon=0.5,

          # ----- Data Augmentation (code‑specific) -----
          data_augmentation=True,
          aug_rename_prob=0.8,
          aug_format_prob=0.8,

          # ----- Advanced Augmentations (mixup, frequency) -----
          mixup_alpha=1.0,
          low_pass_keep_ratio=0.5,
          freq_consistency_weight=0.5,

          # ----- Tokenisation -----
          max_length=512,

          # ------Toggle Modules-----
          # use_mixcode=True,
          # use_fgm=True,
          # use_fft=True,
          # use_r_drop=True,
          # use_focal=False,
          # use_infonce=False,
          use_swa=False,
          # TEST=False,
      )
      trainer, model, tokenizer = train_pipeline(cfg)


2026-04-28 08:56:48,286 - INFO - ===== Training Configuration =====
2026-04-28 08:56:48,288 - INFO - model_name           : microsoft/graphcodebert-base
2026-04-28 08:56:48,289 - INFO - output_dir           : output_checkpoints/graphcodebert-best/
2026-04-28 08:56:48,290 - INFO - num_epochs           : 2
2026-04-28 08:56:48,290 - INFO - max_steps            : -1
2026-04-28 08:56:48,291 - INFO - batch_size           : 512
2026-04-28 08:56:48,292 - INFO - learning_rate        : 1e-06
2026-04-28 08:56:48,293 - INFO - max_length           : 512
2026-04-28 08:56:48,294 - INFO - num_labels           : 2
2026-04-28 08:56:48,295 - INFO - use_wandb            : True
2026-04-28 08:56:48,295 - INFO - freeze_base          : True
2026-04-28 08:56:48,296 - INFO - loss_type            : r-drop
2026-04-28 08:56:48,297 - INFO - focal_alpha          : 1.0
2026-04-28 08:56:48,298 - INFO - focal_gamma          : 2.0
2026-04-28 08:56:48,299 - INFO - r_drop_alpha         : 6.0
2026-04-28 08:56:48,300 - INFO

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-04-28 08:56:49,450 - INFO - Model loaded successfully
2026-04-28 08:56:49,450 - INFO - Model config - num_labels: 2, hidden_size: 768
2026-04-28 08:56:49,588 - INFO - Model placed on cuda
2026-04-28 08:56:49,593 - INFO - Base model frozen - only classifier head is trainable
2026-04-28 08:56:49,593 - INFO - ===== Model Architecture =====
2026-04-28 08:56:49,595 - INFO - 
RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.3, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=7

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,Macro F1
1050,0.742000,0.668785,0.723000,0.569883,0.556374,0.559702
1100,0.744400,0.668781,0.723000,0.569883,0.556374,0.559702
1150,0.740500,0.668613,0.723000,0.572478,0.559571,0.563074
1200,0.737700,0.670175,0.716000,0.582849,0.579047,0.580735
1250,0.736100,0.671897,0.701000,0.585385,0.593374,0.588421
1300,0.733400,0.671437,0.699000,0.583774,0.592087,0.586856
1350,0.732700,0.671928,0.697000,0.594120,0.609984,0.598662


Hyperparameters saved to output_checkpoints/graphcodebert-best/checkpoint-1050/hyperparams.json
2026-04-28 09:06:22,451 - INFO - ✅ Saved hyperparameters to output_checkpoints/graphcodebert-best/checkpoint-1050/config_hyperparams.json
Hyperparameters saved to output_checkpoints/graphcodebert-best/checkpoint-1100/hyperparams.json
2026-04-28 09:21:53,686 - INFO - ✅ Saved hyperparameters to output_checkpoints/graphcodebert-best/checkpoint-1100/config_hyperparams.json
Hyperparameters saved to output_checkpoints/graphcodebert-best/checkpoint-1150/hyperparams.json
2026-04-28 09:37:25,523 - INFO - ✅ Saved hyperparameters to output_checkpoints/graphcodebert-best/checkpoint-1150/config_hyperparams.json
Hyperparameters saved to output_checkpoints/graphcodebert-best/checkpoint-1200/hyperparams.json
2026-04-28 09:52:57,343 - INFO - ✅ Saved hyperparameters to output_checkpoints/graphcodebert-best/checkpoint-1200/config_hyperparams.json
Hyperparameters saved to output_checkpoints/graphcodebert-best/c

In [ ]:
# import os
# import torch
# import json
# from pathlib import Path
# from typing import List, Optional, Dict, Any
# from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

# def interpolate_models_from_dirs(
#     base_dir: str,
#     output_dir: str,
#     method: str = "average",
#     model_class=AutoModelForSequenceClassification,
#     tokenizer_class=AutoTokenizer,
#     model_subdir: str = "final_model",   # subfolder inside each run containing model files
#     validation_metric: Optional[str] = "macro_f1",  # if using best models, which metric
#     top_k: Optional[int] = None,         # if set, use only top_k models by validation metric
#     device: str = "cpu",
#     save_model: bool = True,
# ) -> str:
#     """
#     Load all model checkpoints from subdirectories inside `base_dir` and interpolate their weights.

#     Args:
#         base_dir: Root directory containing run folders (e.g., run_1, run_2, ...)
#         output_dir: Directory where the interpolated model will be saved
#         method: Interpolation method – only 'average' is implemented (uniform averaging)
#         model_class: HuggingFace model class (default AutoModelForSequenceClassification)
#         tokenizer_class: Tokenizer class
#         model_subdir: Subfolder inside each run containing the model files
#                       (e.g., "final_model" or "checkpoint-xxx")
#         validation_metric: Metric name used to select best models (if trainer_state.json exists)
#         top_k: If provided, use only the top_k models sorted by validation_metric (higher is better)
#         device: Device for loading tensors (usually 'cpu' for memory efficiency)
#         save_model: Whether to save the interpolated model and tokenizer
#     Returns:
#         Path to the saved interpolated model directory
#     """
#     base_path = Path(base_dir)
#     output_path = Path(output_dir)
#     output_path.mkdir(parents=True, exist_ok=True)

#     # 1. Collect all valid model directories
#     model_dirs = []
#     run_metrics = []  # (run_dir, metric_value)

#     for run_dir in sorted(base_path.glob("run_*")):
#         model_dir = run_dir / model_subdir
#         if not model_dir.exists():
#             # Fallback: maybe the run directory itself contains the model files
#             model_dir = run_dir
#             if not (model_dir / "pytorch_model.bin").exists() and not (model_dir / "model.safetensors").exists():
#                 continue
#         # Check for trainer_state.json to extract validation metric
#         trainer_state_path = run_dir / "trainer_state.json"
#         metric_val = None
#         if validation_metric and trainer_state_path.exists():
#             try:
#                 with open(trainer_state_path, "r") as f:
#                     state = json.load(f)
#                 # Look for best metric in log_history
#                 best_value = -float("inf")
#                 for entry in state.get("log_history", []):
#                     if f"eval_{validation_metric}" in entry:
#                         val = entry[f"eval_{validation_metric}"]
#                         if val > best_value:
#                             best_value = val
#                 if best_value > -float("inf"):
#                     metric_val = best_value
#             except Exception as e:
#                 print(f"Warning: Could not parse trainer_state.json for {run_dir}: {e}")

#         model_dirs.append(model_dir)
#         run_metrics.append((model_dir, metric_val))

#     if not model_dirs:
#         raise ValueError(f"No valid model directories found under {base_dir}")

#     print(f"Found {len(model_dirs)} model directories.")

#     # 2. Filter by top_k based on validation metric (if available)
#     if top_k and top_k > 0 and validation_metric:
#         # Sort by metric descending, put None at the end
#         run_metrics.sort(key=lambda x: x[1] if x[1] is not None else -float("inf"), reverse=True)
#         selected = [item for item in run_metrics if item[1] is not None][:top_k]
#         if not selected:
#             print(f"No runs with {validation_metric} found. Using all runs.")
#             model_dirs = [item[0] for item in run_metrics]
#         else:
#             model_dirs = [item[0] for item in selected]
#             print(f"Selected top {len(model_dirs)} models based on {validation_metric}.")
#     else:
#         print(f"Using all {len(model_dirs)} models.")


#     # 3. Load reference model architecture (from first model)
#     first_model_dir = model_dirs[0]
#     config = AutoConfig.from_pretrained(first_model_dir)
#     ref_model = model_class.from_config(config).to(device)
#     ref_state_dict = ref_model.state_dict()

#     # 4. Collect state dicts from all models
#     state_dicts = []
#     for model_dir in model_dirs:
#         # Load model weights (safe to load on CPU)
#         try:
#             # Try to load using from_pretrained (handles safetensors and bin)
#             model = model_class.from_pretrained(model_dir, config=config).to(device)
#             state_dicts.append(model.state_dict())
#         except Exception as e:
#             print(f"Error loading model from {model_dir}: {e}")
#             continue

#     if not state_dicts:
#         raise RuntimeError("No models could be loaded.")

#     # 5. Interpolate weights (simple average)
#     print(f"Interpolating {len(state_dicts)} models using method='{method}'...")
#     fused_state = {}
#     for key in ref_state_dict.keys():
#         if key not in state_dicts[0]:
#             # Some models may have extra keys (e.g., classifier weights differ)
#             # Skip or handle by using ref_model's key if present
#             continue
#         # Stack all tensors for this key
#         stacked = torch.stack([sd[key].float() for sd in state_dicts], dim=0)
#         if method == "average":
#             fused = stacked.mean(dim=0)
#         else:
#             raise ValueError(f"Method '{method}' not implemented. Use 'average'.")
#         fused_state[key] = fused.to(ref_state_dict[key].dtype)

#     # 6. Load fused weights into reference model
#     ref_model.load_state_dict(fused_state, strict=False)

#     # 7. Save interpolated model and tokenizer
#     if save_model:
#         print(f"Saving interpolated model to {output_path}")
#         ref_model.save_pretrained(output_path)
#         # Save tokenizer from the first model
#         tokenizer = tokenizer_class.from_pretrained(first_model_dir)
#         tokenizer.save_pretrained(output_path)
#         print(f"Model and tokenizer saved to {output_path}")
#     else:
#         print("Model not saved (save_model=False).")

#     return str(output_path)

# # Example usage:
# if __name__ == "__main__":
#     # After running the sweep, interpolate all models saved in run_*/final_model
#     interpolated_model_dir = interpolate_models_from_dirs(
#         base_dir="output_checkpoints/graphcodebert-base-lowLR-highBatchSize/checkpoint-350",
#         output_dir="output_checkpoints/graphcodebert-ready-lmc/interpolated_model",
#         method="average",
#         model_subdir="final_model",
#         validation_metric="macro_f1",  # assumes we have trainer_state.json
#         top_k=10,                      # use only top 10 models by validation F1
#         device="cpu",                  # CPU is fine for weight averaging
#         save_model=True,
#     )
#     print(f"Interpolated model available at: {interpolated_model_dir}")

In [ ]:
import os
import zipfile

print(f"Searching for zip files in: {OUTPUT_PATH}")

for filename in os.listdir(OUTPUT_PATH):
    if filename.endswith(".zip"):
        filepath = os.path.join(OUTPUT_PATH, filename)
        try:
            with zipfile.ZipFile(filepath, 'r') as zip_ref:
                print(f"Extracting {filename} to {OUTPUT_PATH}...")
                zip_ref.extractall(OUTPUT_PATH)
            print(f"Successfully extracted {filename}.")
            # Optionally, remove the zip file after extraction
            # os.remove(filepath)
            # print(f"Removed {filename}.")
        except zipfile.BadZipFile:
            print(f"Warning: {filename} is a bad zip file and cannot be extracted.")
        except Exception as e:
            print(f"Error extracting {filename}: {e}")

print("Zip file extraction process completed.")

In [ ]:
import os
import json
import logging
from pathlib import Path
from typing import Tuple, Dict, Any
import numpy as np
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


def log_model_architecture_inference(model, tokenizer, logger):
    logger.info("===== Model Architecture =====")
    logger.info("\n" + model.__repr__())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    logger.info("===== Parameter Summary =====")
    logger.info(f"Total Parameters:         {total_params:,}")
    logger.info(f"Trainable Parameters:     {trainable_params:,}")
    logger.info(f"Non-trainable Parameters: {non_trainable_params:,}")
    logger.info("===== Tokenizer Summary =====")
    logger.info(f"Vocab size: {len(tokenizer)} | Special tokens: {tokenizer.all_special_tokens}")
    logger.info("===== End of Architecture Log =====")


def run_standalone_inference(
    checkpoint_path: str,
    output_dir: str = "./",
    output_csv: str = "submission.csv",
    batch_size: int = 32,
    max_length: int = 512,
    dataset_name: str = "DaniilOr/SemEval-2026-Task13",
    dataset_config: str = "A",
    split: str = "test",
):
    # === derive checkpoint subfolder ===
    checkpoint_name = Path(checkpoint_path).name
    save_dir = Path(output_dir) / checkpoint_name
    save_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logger(save_dir, "inference")
    logger.info(f"Loading model and tokenizer from: {checkpoint_path}")

    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)

    log_model_architecture_inference(model, tokenizer, logger)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    # ========== MEMORY-EFFICIENT DATASET LOADING ==========
    logger.info(f"Loading dataset from: {dataset_name}")

    # Check if dataset_name points to a .parquet file (case-insensitive)
    if ".parquet" in dataset_name.lower():
        logger.info("Detected .parquet file – loading directly with datasets (memory-mapped)")
        try:
            from datasets import load_dataset
        except ImportError:
            raise ImportError("datasets library is required to load .parquet files efficiently")

        # Load Parquet file using Hugging Face's native parquet loader
        # This uses memory mapping and Arrow, never loading the whole file into RAM
        parquet_dataset = load_dataset("parquet", data_files=dataset_name, split="train")
        test_ds = parquet_dataset

        logger.info(f"Loaded Parquet file with {len(test_ds)} examples (memory-mapped)")
        logger.info(f"Columns found: {test_ds.column_names}")
        print(test_ds[:10])

        # Warn that the 'split' argument is ignored for local Parquet files
        if split != "test":
            logger.warning(f"Ignoring 'split={split}' – loading full Parquet file instead.")
    else:
        # Original Hugging Face dataset loading logic (from hub or other formats)
        logger.info(f"Loading from Hugging Face hub: {dataset_name} ({dataset_config})")
        try:
            test_ds = load_dataset(dataset_name, dataset_config, split=split)
        except Exception as e:
            logger.warning(f"Default loading failed: {e}")
            logger.info(f"Attempting to load split '{split}' using data_files...")
            test_ds = load_dataset(dataset_name, data_files={split: f"*{split}*"}, split=split)

    # ========== REST OF THE PIPELINE (unchanged) ==========
    def tokenize_fn(examples):
        return tokenizer(
            examples["code"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    logger.info("Tokenizing dataset...")
    tokenized_ds = test_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=[c for c in test_ds.column_names if c not in ["input_ids", "attention_mask", "id", "label"]],
        desc="Tokenizing"
    )
    tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    test_loader = torch.utils.data.DataLoader(
        tokenized_ds,
        batch_size=batch_size,
        shuffle=False,
    )

    logger.info(f"Running inference on {len(test_ds)} examples...")
    all_logits = []

    with torch.inference_mode():
        for batch in tqdm(test_loader, desc="Predicting"):
            inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
            outputs = model(**inputs)
            all_logits.append(outputs.logits.cpu())

    logits = torch.cat(all_logits, dim=0).numpy()
    pred_labels = logits.argmax(axis=-1)

    # # === DUMMY INFERENCE: random label generation ===
    # import numpy as np
    # from tqdm import tqdm
    
    # # Determine number of classes (if unknown, assume 2 or infer from test_ds)
    # # Common for SemEval tasks: often 2 (binary) or 3+ (multi-class). 
    # # Adjust num_classes as needed.
    # num_classes = 2  # change this to your actual number of classes
    
    # all_pred_labels = []
    
    # # Simulate batching: loop over test_loader without using a model
    # with torch.inference_mode():  # optional, not needed but keeps style
    #     for batch in tqdm(test_loader, desc="Generating random predictions"):
    #         batch_size = batch["input_ids"].size(0)  # actual batch size (last batch may be smaller)
    #         # Generate random logits (or directly random labels)
    #         random_logits = torch.randn(batch_size, num_classes)
    #         batch_preds = random_logits.argmax(dim=-1)  # shape: (batch_size,)
    #         all_pred_labels.append(batch_preds.cpu())
    
    # # Concatenate all batch predictions into a single numpy array
    # pred_labels = torch.cat(all_pred_labels, dim=0).numpy()
    


    metrics = {}

    if "label" in test_ds.column_names:
        logger.info("Calculating classification metrics...")
        true_labels = np.array(test_ds["label"])

        acc = accuracy_score(true_labels, pred_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, pred_labels, average='weighted'
        )
        cm = confusion_matrix(true_labels, pred_labels)

        metrics = {
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "confusion_matrix": cm.tolist(),
        }

        logger.info("-" * 30)
        logger.info(f"METRICS FOR SPLIT: {split}")
        logger.info(f"Accuracy:  {acc:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall:    {recall:.4f}")
        logger.info(f"F1-Score:  {f1:.4f}")
        logger.info("-" * 30)
        logger.info(f"Confusion Matrix:\n{cm}")

        metrics_path = save_dir / "metrics.json"
        with open(metrics_path, "w") as f:
            json.dump(metrics, f, indent=4)
        logger.info(f"✅ Metrics saved to {metrics_path}")

    else:
        logger.warning("No 'label' column found. Skipping metric calculation.")

    csv_path = save_dir / output_csv

    if "id" in test_ds.column_names:
        ids = test_ds["id"]
    elif "ID" in test_ds.column_names:
        ids = test_ds["ID"]
    else:
        ids = list(range(len(pred_labels)))

    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("ID,label\n")
        for idx, label in zip(ids, pred_labels):
            f.write(f"{idx},{label}\n")

    logger.info(f"✅ Predictions saved to {csv_path}")

In [ ]:
if __name__ == "__main__":
    import gc
    import torch
    gc.collect()
    if torch.cuda.is_available():
      torch.cuda.empty_cache()

    CHECKPOINT = "checkpoints/graphcodebert-base-lowLR-highBatchSize/checkpoint-1022"
    OUTPUT_DIR = "test/inference/graphcodebert-base-lowLR-highBatchSize"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if os.path.exists(CHECKPOINT):
        run_standalone_inference(
            checkpoint_path=CHECKPOINT,
            output_dir=OUTPUT_DIR,
            output_csv="checkpoint-1022-submission.csv",
            batch_size=32,
            # dataset_name="dzungpham/SemEval-2026-TaskA-dataset",
            # dataset_config="default",
            # dataset_name="DaniilOr/SemEval-2026-Task13",
            # dataset_config="A",
            dataset_name="/kaggle/input/datasets/dzung271828/semeval/Task_A/test.parquet",
            split="test"
        )

In [ ]:
# from huggingface_hub import HfApi

# api = HfApi()
# repo_id = "dzungpham/graphcodebert-code-classification"
# local_folder_path = "output_checkpoints"

# api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# api.upload_folder(
#     folder_path=local_folder_path,
#     repo_id=repo_id,
#     repo_type="model",
#     commit_message="upload checkpoints"
# )

# print(f"✅ Upload complete! View at: https://huggingface.co/{repo_id}")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "dzungpham/graphcodebert-code-classification"
local_folder_path = "test"

api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

api.upload_folder(
    folder_path=local_folder_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="upload inference results"
)

print(f"✅ Upload complete! View at: https://huggingface.co/{repo_id}")

In [ ]:
# =============================================================================
# Hyperparameter sweep (100 steps each, fixed seed) + resumable LMC fusion
# =============================================================================
# References:
# - Garipov et al., "Loss Surfaces, Mode Connectivity, and Fast Ensembling of DNNs" (NeurIPS 2018)
# - Draxler et al., "Essentially No Barriers in Neural Network Energy Landscape" (ICML 2018)
#
# Resumability design:
# - Persist sweep progress to JSON state after every completed run.
# - Keep a rolling fused model checkpoint under latest_fused_model/.
# - Resume from latest fused model for the next hyperparameter combination.
#
# Data efficiency design:
# - Pre-tokenize train/val ONCE outside the sweep loop.
# - Reuse those tokenized datasets across all hyperparameter combinations.

from copy import deepcopy
from itertools import product
from typing import List, Dict, Optional, Tuple
import json


MODEL_DROPOUT_KEYS = {
    "hidden_dropout_prob",
    "attention_probs_dropout_prob",
    "classifier_dropout",
}


def _build_compute_loss_fn(cfg: TrainConfig):
    if cfg.loss_type == "focal":
        return get_focal_loss(
            cfg.focal_alpha,
            cfg.focal_gamma,
            smoothing=cfg.label_smoothing,
        )
    if cfg.loss_type == "infonce":
        return get_infonce_loss(cfg.infonce_temperature, cfg.infonce_weight)
    return None


def _save_json(path: str, payload: Dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def _load_json(path: str) -> Dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _load_model_and_tokenizer_from_source(
    model_source: str,
    num_labels: int,
):
    tokenizer = AutoTokenizer.from_pretrained(model_source)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_source,
        num_labels=num_labels,
        problem_type="single_label_classification",
    )
    return model, tokenizer


def _prepare_tokenized_datasets_once(base_cfg: TrainConfig) -> Tuple[Dataset, Dataset]:
    """
    Build tokenized train/val exactly once and reuse for all runs.
    NOTE: per-run augmentation probabilities will not change tokenized data when using this path.
    """
    print("Preparing tokenized datasets once for the entire sweep...")
    tokenizer = AutoTokenizer.from_pretrained(base_cfg.model_name)

    # Pre-tokenize once. We intentionally disable per-run augmentation remapping here.
    train_dataset, val_dataset = load_datasets(tokenizer, max_length=base_cfg.max_length, aug=None)

    print(f"Tokenized datasets ready. Train={len(train_dataset)} | Val={len(val_dataset)}")
    return train_dataset, val_dataset


def _apply_dropout_overrides(model, overrides: Dict):
    hidden_p = overrides.get("hidden_dropout_prob")
    attention_p = overrides.get("attention_probs_dropout_prob")
    classifier_p = overrides.get("classifier_dropout")

    if hidden_p is not None and hasattr(model.config, "hidden_dropout_prob"):
        model.config.hidden_dropout_prob = hidden_p
    if attention_p is not None and hasattr(model.config, "attention_probs_dropout_prob"):
        model.config.attention_probs_dropout_prob = attention_p
    if classifier_p is not None and hasattr(model.config, "classifier_dropout"):
        model.config.classifier_dropout = classifier_p

    # Apply dropout rates directly to instantiated modules.
    if hasattr(model, "roberta"):
        roberta = model.roberta

        if hidden_p is not None and hasattr(roberta.embeddings, "dropout"):
            roberta.embeddings.dropout.p = float(hidden_p)

        if hasattr(roberta, "encoder") and hasattr(roberta.encoder, "layer"):
            for layer in roberta.encoder.layer:
                if attention_p is not None and hasattr(layer.attention.self, "dropout"):
                    layer.attention.self.dropout.p = float(attention_p)
                if hidden_p is not None and hasattr(layer.attention.output, "dropout"):
                    layer.attention.output.dropout.p = float(hidden_p)
                if hidden_p is not None and hasattr(layer.output, "dropout"):
                    layer.output.dropout.p = float(hidden_p)

    if classifier_p is not None and hasattr(model, "classifier"):
        if hasattr(model.classifier, "dropout"):
            model.classifier.dropout.p = float(classifier_p)


def _train_single_config_for_steps(
    base_cfg: TrainConfig,
    overrides: Dict,
    run_output_dir: str,
    fixed_seed: int,
    max_steps_per_config: int,
    save_steps_per_config: int,
    train_dataset: Dataset,
    val_dataset: Dataset,
    init_model_source: Optional[str] = None,
):
    cfg = deepcopy(base_cfg)
    cfg.output_dir = run_output_dir
    cfg.seed = fixed_seed

    for key, value in overrides.items():
        if key in MODEL_DROPOUT_KEYS:
            continue
        if not hasattr(cfg, key):
            raise KeyError(f"Unknown TrainConfig field: {key}")
        setattr(cfg, key, value)

    set_seed(fixed_seed)
    os.makedirs(cfg.output_dir, exist_ok=True)

    local_logger = setup_logger(cfg.output_dir, "sweep_train")
    local_logger.info("Starting config override: %s", overrides)
    local_logger.info("Fixed seed: %d | Max steps: %d", fixed_seed, max_steps_per_config)

    source = init_model_source if init_model_source else cfg.model_name
    local_logger.info("Model init source: %s", source)
    model, tokenizer = _load_model_and_tokenizer_from_source(source, cfg.num_labels)
    _apply_dropout_overrides(model, overrides)
    model.to(cfg.device)

    if cfg.freeze_base:
        freeze_base_model(model)

    if cfg.loss_type == "infonce":
        model.config.output_hidden_states = True

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        max_steps=max_steps_per_config,
        num_train_epochs=max(cfg.num_epochs, 1),
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size * 2,
        learning_rate=cfg.learning_rate,
        warmup_steps=max(1, int(max_steps_per_config * 0.1)),
        weight_decay=0.1,
        logging_strategy="steps",
        logging_steps=2,
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=max(1, min(save_steps_per_config, max_steps_per_config)),
        save_total_limit=2,
        report_to=["wandb"] if cfg.use_wandb else [],
        seed=fixed_seed,
        dataloader_pin_memory=torch.cuda.is_available(),
        fp16=torch.cuda.is_available(),
        bf16=False,
    )

    trainer = RobustTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics_fn,
        callbacks=[ConfigLoggingCallback(train_config=cfg)],
        loss_type=cfg.loss_type,
        r_drop_alpha=cfg.r_drop_alpha,
        compute_loss_fn=_build_compute_loss_fn(cfg),
        label_smoothing=cfg.label_smoothing,
        adversarial_epsilon=cfg.adversarial_epsilon,
        use_swa=cfg.use_swa,
        swa_start_epoch=cfg.swa_start_epoch,
        swa_lr=cfg.swa_lr,
        mixup_alpha=cfg.mixup_alpha,
        low_pass_keep_ratio=cfg.low_pass_keep_ratio,
        freq_consistency_weight=cfg.freq_consistency_weight,
    )

    trainer.train(resume_from_checkpoint=None)

    final_dir = os.path.join(cfg.output_dir, "final_model")
    os.makedirs(final_dir, exist_ok=True)
    if trainer._swa_started and trainer.swa_model is not None:
        model_to_save = trainer.swa_model.module if hasattr(trainer.swa_model, "module") else trainer.swa_model
    else:
        model_to_save = trainer.model

    model_to_save.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    local_logger.info("Saved final model to: %s", final_dir)

    return final_dir, overrides


def fuse_models_with_lmc(
    model_dirs: List[str],
    output_dir: str,
    interpolation_control: float = 0.5,
):
    if len(model_dirs) < 2:
        raise ValueError("Need at least 2 trained models for LMC fusion.")
    if not (0.0 <= interpolation_control <= 1.0):
        raise ValueError("interpolation_control must be in [0, 1].")

    os.makedirs(output_dir, exist_ok=True)

    base_model = AutoModelForSequenceClassification.from_pretrained(model_dirs[0])
    fused_state = {
        key: value.detach().clone()
        for key, value in base_model.state_dict().items()
    }

    for next_dir in model_dirs[1:]:
        next_state = AutoModelForSequenceClassification.from_pretrained(next_dir).state_dict()
        if set(next_state.keys()) != set(fused_state.keys()):
            raise ValueError(f"State dict keys mismatch between models: {model_dirs[0]} and {next_dir}")

        for key in fused_state:
            if fused_state[key].is_floating_point():
                fused_state[key] = (
                    (1.0 - interpolation_control) * fused_state[key]
                    + interpolation_control * next_state[key]
                )

    base_model.load_state_dict(fused_state, strict=True)
    tokenizer = AutoTokenizer.from_pretrained(model_dirs[0])
    base_model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    return output_dir


def fuse_pair_with_lmc(
    base_model_dir: str,
    new_model_dir: str,
    output_dir: str,
    interpolation_control: float,
):
    return fuse_models_with_lmc(
        model_dirs=[base_model_dir, new_model_dir],
        output_dir=output_dir,
        interpolation_control=interpolation_control,
    )


def build_hyperparam_sets_from_grid(param_grid: Dict[str, List], max_combinations: int = 288) -> List[Dict]:
    """
    Build hyperparameter dictionaries from Cartesian product of per-parameter values.
    If the full grid is larger than max_combinations, keep a deterministic evenly spaced subset.
    """
    grid_keys = list(param_grid.keys())
    value_lists = [param_grid[key] for key in grid_keys]

    for key, values in param_grid.items():
        if not values:
            raise ValueError(f"Parameter '{key}' has an empty value list.")

    all_combos = list(product(*value_lists))
    total = len(all_combos)

    if total <= max_combinations:
        selected = all_combos
    else:
        step = (total - 1) / (max_combinations - 1)
        selected_indices = [int(round(i * step)) for i in range(max_combinations)]
        seen = set()
        unique_indices = []
        for idx in selected_indices:
            if idx not in seen:
                seen.add(idx)
                unique_indices.append(idx)
        selected = [all_combos[idx] for idx in unique_indices]

    hyperparam_sets = [
        {key: combo[i] for i, key in enumerate(grid_keys)}
        for combo in selected
    ]

    print(f"Total Cartesian combinations: {total}")
    print(f"Using combinations: {len(hyperparam_sets)} (max allowed: {max_combinations})")
    return hyperparam_sets


def run_hparam_sweep_and_lmc_resumable(
    base_cfg: TrainConfig,
    hyperparam_sets: List[Dict],
    sweep_output_dir: str,
    fixed_seed: int = 42,
    max_steps_per_config: int = 100,
    save_steps_per_config: int = 50,
    interpolation_control: float = 0.5,
    resume: bool = True,
):
    """
    Resumable hyperparameter sweep.

    Behavior:
    - Persists progress to sweep_state.json after each completed combination.
    - Starts each new run from the latest fused model if available.
    - Produces final fused model after all combinations are processed.
    - Reuses pre-tokenized train/val datasets built once outside the loop.
    """
    if not hyperparam_sets:
        raise ValueError("Provide at least one hyperparameter set.")

    os.makedirs(sweep_output_dir, exist_ok=True)
    state_path = os.path.join(sweep_output_dir, "sweep_state.json")
    latest_fused_dir = os.path.join(sweep_output_dir, "latest_fused_model")
    final_fused_dir = os.path.join(sweep_output_dir, "final_fused_model")

    if resume and os.path.exists(state_path):
        state = _load_json(state_path)
        if state.get("total_combinations") != len(hyperparam_sets):
            raise ValueError(
                "Existing state total_combinations does not match current hyperparam_sets length. "
                "Either keep the same grid or start with a new sweep_output_dir."
            )
        print("Resuming from existing state file.")
    else:
        state = {
            "version": 3,
            "total_combinations": len(hyperparam_sets),
            "next_index": 0,
            "trained_model_dirs": [],
            "latest_fused_model_dir": None,
            "final_fused_model_dir": None,
            "fixed_seed": fixed_seed,
            "max_steps_per_config": max_steps_per_config,
            "save_steps_per_config": save_steps_per_config,
            "interpolation_control": interpolation_control,
            "completed": False,
        }
        _save_json(state_path, state)
        print("Initialized new sweep state file.")

    # Pre-tokenize once and reuse for all runs.
    train_dataset, val_dataset = _prepare_tokenized_datasets_once(base_cfg)

    while state["next_index"] < len(hyperparam_sets):
        run_idx = state["next_index"] + 1
        overrides = hyperparam_sets[state["next_index"]]
        run_dir = os.path.join(sweep_output_dir, f"run_{run_idx:03d}")

        init_source = state.get("latest_fused_model_dir") or base_cfg.model_name
        print(f"[{run_idx}/{len(hyperparam_sets)}] init from: {init_source}")

        final_dir, used_overrides = _train_single_config_for_steps(
            base_cfg=base_cfg,
            overrides=overrides,
            run_output_dir=run_dir,
            fixed_seed=fixed_seed,
            max_steps_per_config=max_steps_per_config,
            save_steps_per_config=save_steps_per_config,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            init_model_source=init_source,
        )

        trained_model_dirs = state.get("trained_model_dirs", [])
        trained_model_dirs.append(final_dir)
        state["trained_model_dirs"] = trained_model_dirs

        if state.get("latest_fused_model_dir") is None:
            model_anchor = AutoModelForSequenceClassification.from_pretrained(final_dir)
            tokenizer_anchor = AutoTokenizer.from_pretrained(final_dir)
            os.makedirs(latest_fused_dir, exist_ok=True)
            model_anchor.save_pretrained(latest_fused_dir)
            tokenizer_anchor.save_pretrained(latest_fused_dir)
        else:
            fuse_pair_with_lmc(
                base_model_dir=state["latest_fused_model_dir"],
                new_model_dir=final_dir,
                output_dir=latest_fused_dir,
                interpolation_control=interpolation_control,
            )

        state["latest_fused_model_dir"] = latest_fused_dir
        state["next_index"] += 1
        state["completed"] = state["next_index"] >= len(hyperparam_sets)
        _save_json(state_path, state)

        print(f"Completed run {run_idx:03d}. Overrides: {used_overrides}")
        print(f"Progress saved to: {state_path}")

    if state.get("latest_fused_model_dir"):
        final_model = AutoModelForSequenceClassification.from_pretrained(state["latest_fused_model_dir"])
        final_tokenizer = AutoTokenizer.from_pretrained(state["latest_fused_model_dir"])
        os.makedirs(final_fused_dir, exist_ok=True)
        final_model.save_pretrained(final_fused_dir)
        final_tokenizer.save_pretrained(final_fused_dir)
        state["final_fused_model_dir"] = final_fused_dir
        state["completed"] = True
        _save_json(state_path, state)

    print("Sweep complete.")
    print(f"Final fused model: {state.get('final_fused_model_dir')}")

    return state.get("trained_model_dirs", []), state.get("final_fused_model_dir")

In [ ]:
# # -----------------------------------------------------------------------------
# # Base config aligned to your 2026-04-20 run, then vicinity grid around it.
# # -----------------------------------------------------------------------------

# BASE_CFG = TrainConfig(
#     model_name="microsoft/graphcodebert-base",
#     output_dir="output_checkpoints/graphcodebert_lmc_sweep_v2",
#     num_epochs=2,
#     batch_size=64,
#     learning_rate=5e-5,
#     max_length=512,
#     use_wandb=True,
#     freeze_base=True,
#     loss_type="r-drop",
#     r_drop_alpha=6.0,
#     label_smoothing=0.3,
#     adversarial_epsilon=0.5,
#     use_swa=True,
#     swa_start_epoch=1,
#     swa_lr=1e-5,
#     data_augmentation=True,
#     aug_rename_prob=0.6,
#     aug_format_prob=0.6,
#     # New parameters (best checkpoints 200 curently dont use these ones yet)
#     mixup_alpha=1.0,
#     low_pass_keep_ratio=0.5,
#     freq_consistency_weight=0.2,

#     hidden_dropout_prob = 0.3,
#     attention_probs_dropout_prob = 0.3,
#     classifier_dropout = 0.3,
# )

# PARAM_GRID = {
#     # Batch size – affects gradient noise and convergence speed
#     "batch_size": [48, 64, 80],

#     # Learning rate – most critical for head training
#     "learning_rate": [3e-5, 5e-5, 7e-5],

#     # R-Drop strength – controls consistency between two forward passes
#     "r_drop_alpha": [5.0, 6.0, 7.0],

#     # Label smoothing – prevents overconfidence, useful for LMC
#     "label_smoothing": [0.2, 0.3, 0.4],

#     # MixCode mixing strength – augments embeddings
#     "mixup_alpha": [0.8, 1.0, 1.2],

#     # Dropout in the classifier head (most trainable part)
#     "classifier_dropout": [0.2, 0.3, 0.4],
# }
# # Keep selected combinations manageable.
# MAX_SWEEP_COMBINATIONS = 10_000
# HYPERPARAM_SETS = build_hyperparam_sets_from_grid(
#     param_grid=PARAM_GRID,
#     max_combinations=MAX_SWEEP_COMBINATIONS,
# )

# INTERPOLATION_CONTROL = 0.5
# SAVE_STEPS_PER_CONFIG = 100

# trained_dirs, fused_model_dir = run_hparam_sweep_and_lmc_resumable(
#     base_cfg=BASE_CFG,
#     hyperparam_sets=HYPERPARAM_SETS,
#     sweep_output_dir="output_checkpoints/graphcodebert_lmc_sweep_v2",
#     fixed_seed=42,
#     max_steps_per_config=100,
#     save_steps_per_config=SAVE_STEPS_PER_CONFIG,
#     interpolation_control=INTERPOLATION_CONTROL,
#     resume=True,
# )